# 📊 Notebook 01: CCTV Dataset Exploration, Video Characterization & Multi-Camera Analysis
### Capstone: CLIP vs. V-JEPA for CCTV Event Retrieval & Hallucination Analysis
---
**Objective:** Perform comprehensive exploratory data analysis (EDA) across the **4 real-world surveillance datasets (5,153 clips)**:
1. **RLVS:** 2,000 clips (*Violence vs. NonViolence*)
2. **UCF-ARG:** 1,410 clips across 3 camera perspectives (*Aerial, Ground, Rooftop*)
3. **TinyVIRAT-v2:** 743 clips (*Vehicles, Carrying, Tools, Interactions*)
4. **UCF-Crime:** 1,000 clips across 14 anomaly classes (*Robbery, Arson, Accidents, Shoplifting, etc.*)

We examine class distributions, video duration and resolution distributions, camera viewpoint imbalances, and visual domain shifts.


## Step 1: Ingest Unified Dataset Metadata
Load pre-indexed metadata and feature registers across all 4 datasets.


In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import cv2

NOTEBOOK_DIR = Path.cwd()
REPO_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == "CLIP_vs_VJEPA_CCTV_Event_Retrieval_and_Hallucination_Analysis" else NOTEBOOK_DIR
sys.path.append(str(REPO_ROOT / "notebooks"))
sys.path.append(str(REPO_ROOT / "implementation"))
sys.path.append(str(REPO_ROOT))

import notebook_utils as nu

visual_embs, metadata = nu.load_all_datasets_data()
df = pd.DataFrame(metadata)

print(f"[Dataset EDA] Loaded {len(df):,} total clips.")
print(f"[Dataset EDA] Feature Vector Dimensions: {visual_embs.shape[1]}")
display(df.head(5))


## Step 2: Cross-Dataset Volume & Class Distributions
Analyze the distribution of video clips across the 4 benchmarks.


In [ ]:
plt.figure(figsize=(10, 4.5))
palette = ["#1f77b4", "#ff7f0e", "#d62728", "#2ca02c"]
ax = sns.countplot(data=df, x="dataset", order=df["dataset"].value_counts().index, palette=palette, edgecolor="#333")
plt.title("Total CCTV Video Clips by Dataset Benchmark", fontsize=14, fontweight="bold")
plt.xlabel("Dataset", fontweight="bold")
plt.ylabel("Number of Videos", fontweight="bold")
plt.xticks(range(4), [d.upper() for d in df["dataset"].value_counts().index])

for p in ax.patches:
    h = p.get_height()
    ax.annotate(f"{int(h):,} ({h/len(df)*100:.1f}%)", 
                (p.get_x() + p.get_width() / 2., h / 2),
                ha='center', va='center', color='white', fontweight='bold', fontsize=11)

plt.tight_layout()
plt.show()

print("\nClass Category Distribution Breakdown:")
print(df.groupby(["dataset", "label"]).size().to_string())


## Step 3: Camera Viewpoint Taxonomy Analysis
Surveillance events exhibit severe perspective shifts. Analyze the distribution of camera viewpoints (Aerial, Ground, Rooftop) across UCF-ARG and other datasets.


In [ ]:
def extract_viewpoint(row):
    meta = row.get("metadata", {})
    if isinstance(meta, dict):
        if "viewpoint" in meta:
            return meta["viewpoint"].lower()
    return "fixed_cctv"

df["viewpoint_category"] = df.apply(extract_viewpoint, axis=1)

plt.figure(figsize=(9, 4))
vp_counts = df["viewpoint_category"].value_counts()
vp_counts.plot(kind="barh", color=["#3b528b", "#21918c", "#5ec962", "#fde725"], edgecolor="#222")
plt.title("Surveillance Camera Viewpoint Distribution", fontweight="bold", fontsize=13)
plt.xlabel("Clip Count", fontweight="bold")
plt.ylabel("Camera Viewpoint", fontweight="bold")

for idx, val in enumerate(vp_counts):
    plt.text(val + 30, idx, f"{val:,} ({val/len(df)*100:.1f}%)", va='center', fontweight='bold')

plt.xlim(0, max(vp_counts) * 1.2)
plt.tight_layout()
plt.show()


## Step 4: Video Duration & Frame Count Characterization
Audit video duration, frame counts, and frame rates using OpenCV metadata extraction across sampled clips.


In [ ]:
sample_df = df.sample(n=min(300, len(df)), random_state=42)
durations = []
f_counts = []
fps_list = []

for _, row in sample_df.iterrows():
    p = row["path"]
    if Path(p).exists():
        cap = cv2.VideoCapture(p)
        fc = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
        durations.append(fc / fps if fps > 0 else 0)
        f_counts.append(fc)
        fps_list.append(fps)
        cap.release()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))
sns.histplot(durations, bins=25, kde=True, color="#1f77b4", ax=ax1)
ax1.set_title("Clip Duration Distribution (Seconds)", fontweight="bold")
ax1.set_xlabel("Duration (s)", fontweight="bold")

sns.histplot(fps_list, bins=15, kde=False, color="#2ca02c", ax=ax2)
ax2.set_title("Surveillance Video Frame Rates (FPS)", fontweight="bold")
ax2.set_xlabel("FPS", fontweight="bold")

plt.tight_layout()
plt.show()

print(f"Mean Duration: {np.mean(durations):.2f}s | Median: {np.median(durations):.2f}s | Max: {np.max(durations):.2f}s")


## Step 5: Multi-Dataset Visual Keyframe Grid
Display authentic video keyframe samples across the four benchmarks to visualize domain shifts, lighting differences, and surveillance challenges.


In [ ]:
fig, axes = plt.subplots(4, 4, figsize=(16, 11))
datasets_order = ["rlvs", "ucf_arg", "tinyvirat", "ucf_crime"]

for row_idx, ds_name in enumerate(datasets_order):
    sub = df[df["dataset"] == ds_name].sample(n=4, random_state=42)
    for col_idx, (_, item) in enumerate(sub.iterrows()):
        frames = nu.extract_video_keyframes(item["path"], num_frames=1)
        ax = axes[row_idx, col_idx]
        if frames:
            ax.imshow(frames[0])
            lbl = item["label"]
            ax.set_title(f"[{ds_name.upper()}] {lbl[:20]}", fontsize=9.5, fontweight="bold")
        else:
            ax.text(0.5, 0.5, "Frame Unavailable", ha='center', va='center')
        ax.axis("off")

plt.suptitle("Multi-Dataset CCTV Visual Benchmark Grid (RLVS, UCF-ARG, TinyVIRAT, UCF-Crime)", fontsize=14, fontweight="bold", y=0.99)
plt.tight_layout()
plt.show()


## Step 6: Summary of Exploratory Findings
- **Scale:** 5,153 real surveillance videos represent diverse urban environments, multi-camera angles, and complex human-vehicle interactions.
- **Viewpoint Disparity:** Aerial and Rooftop cameras produce acute perspective foreshortening that challenges 2D spatial models.
- **Class Imbalance:** Road accidents and robberies predominate anomaly benchmarks, requiring stratified sampling during training and validation.
